In [1]:
import pickle

with open("data/train_processed.pkl", "rb") as f:
    train_processed = pickle.load(f)

with open("data/val_processed.pkl", "rb") as f:
    val_processed = pickle.load(f)

with open("data/test_processed.pkl", "rb") as f:
    test_processed = pickle.load(f)

with open("data/word2idx_2.pkl", "rb") as f:
    word2idx_2 = pickle.load(f)

with open("data/idx2word_2.pkl", "rb") as f:
    idx2word_2 = pickle.load(f)

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader

import numpy as np

from scipy.io import loadmat

In [3]:
class SARVQADataset(Dataset):

    def __init__(self, data):

        self.data = data

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        sample = self.data[idx]

        filepath = sample["filepath"]

        mat = loadmat(filepath)

        complex_img = mat["complex_img"]

        real = complex_img.real
        imag = complex_img.imag

        image = np.stack(
            [real, imag],
            axis=0
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        )

        question = torch.tensor(
            sample["question_tokens"],
            dtype=torch.long
        )

        answer = torch.tensor(
            sample["answer"],
            dtype=torch.long
        )

        return image, question, answer

In [4]:
train_dataset = SARVQADataset(
    train_processed
)

image, question, answer = train_dataset[0]

print(image.shape)
print(question.shape)
print(answer)

torch.Size([2, 128, 128])
torch.Size([5])
tensor(1)


In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [6]:
images, questions, answers = next(
    iter(train_loader)
)

print(images.shape)
print(questions.shape)
print(answers.shape)

torch.Size([32, 2, 128, 128])
torch.Size([32, 5])
torch.Size([32])


Building first RVNN base model

In [7]:
import torch
import torch.nn as nn

class QuestionEncoder(nn.Module):

    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, questions):

        x = self.embedding(questions)

        _, (h, _) = self.lstm(x)

        h_forward = h[-2]
        h_backward = h[-1]

        return torch.cat(
            [h_forward, h_backward],
            dim=1
        )

In [8]:
class SARImageEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(
                2,
                16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                16,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d(1)
        )

    def forward(self, x):

        x = self.cnn(x)

        return x.flatten(1)

In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

In [10]:
class SemanticEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                64,
                48
            ),

            nn.ReLU(),

            nn.Linear(
                48,
                32
            )
        )

    def forward(self, x):

        return self.net(x)

In [11]:
def awgn_channel(
    x,
    snr_db
):

    signal_power = (
        x.pow(2)
        .mean()
    )

    snr_linear = (
        10 ** (snr_db / 10)
    )

    noise_power = (
        signal_power
        / snr_linear
    )

    noise = torch.randn_like(x) * torch.sqrt(
        noise_power
    )

    return x + noise

In [12]:
class SemanticDecoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                32,
                48
            ),

            nn.ReLU(),

            nn.Linear(
                48,
                64
            )
        )

    def forward(self, x):

        return self.net(x)

In [13]:
class RVNNVQA(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.encoder = SemanticEncoder()
        self.decoder = SemanticDecoder()

        self.image_encoder = (
            SARImageEncoder()
        )

        self.question_encoder = (
            QuestionEncoder(
                vocab_size
            )
        )

        self.classifier = nn.Sequential(

            nn.Linear(
                64 + 128,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                64
            ),

            nn.ReLU(),

            nn.Linear(
                64,
                2
            )
        )

    def forward(
        self,
        images,
        questions
    ):

        image_vec = self.image_encoder(
            images
        )

        compressed = self.encoder(
            image_vec
        )

        received = awgn_channel(
            compressed,
            snr_db=10
        )

        reconstructed = self.decoder(
            received
        )

        question_vec = self.question_encoder(
            questions
        )

        fused = torch.cat(
            [
                reconstructed,
                question_vec
            ],
            dim=1
        )

        logits = self.classifier(
            fused
        )

        return logits

In [14]:
model = RVNNVQA(
    vocab_size=len(word2idx_2)
).to(device)

images, questions, answers = next(
    iter(train_loader)
)

images = images.to(device)
questions = questions.to(device)

logits = model(
    images,
    questions
)

print(logits.shape)

torch.Size([32, 2])


In [15]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [16]:
val_dataset = SARVQADataset(
    val_processed
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [17]:
num_epochs = 20

for epoch in range(num_epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, questions, answers in train_loader:

        images = images.to(device)
        questions = questions.to(device)
        answers = answers.to(device)

        optimizer.zero_grad()

        logits = model(
            images,
            questions
        )

        loss = criterion(
            logits,
            answers
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == answers
        ).sum().item()

        total += answers.size(0)

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {total_loss/len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )

Epoch [1/20] Loss: 0.5707 Accuracy: 68.26%
Epoch [2/20] Loss: 0.5153 Accuracy: 74.04%
Epoch [3/20] Loss: 0.4739 Accuracy: 76.00%
Epoch [4/20] Loss: 0.4263 Accuracy: 78.96%
Epoch [5/20] Loss: 0.3688 Accuracy: 81.94%
Epoch [6/20] Loss: 0.3313 Accuracy: 84.70%
Epoch [7/20] Loss: 0.3126 Accuracy: 85.41%
Epoch [8/20] Loss: 0.2996 Accuracy: 85.80%
Epoch [9/20] Loss: 0.2803 Accuracy: 87.08%
Epoch [10/20] Loss: 0.2667 Accuracy: 87.52%
Epoch [11/20] Loss: 0.2688 Accuracy: 87.58%
Epoch [12/20] Loss: 0.2504 Accuracy: 88.03%
Epoch [13/20] Loss: 0.2402 Accuracy: 89.15%
Epoch [14/20] Loss: 0.2197 Accuracy: 90.18%
Epoch [15/20] Loss: 0.2181 Accuracy: 89.80%
Epoch [16/20] Loss: 0.2088 Accuracy: 90.31%
Epoch [17/20] Loss: 0.2225 Accuracy: 89.95%
Epoch [18/20] Loss: 0.1921 Accuracy: 91.24%
Epoch [19/20] Loss: 0.1848 Accuracy: 91.54%
Epoch [20/20] Loss: 0.1712 Accuracy: 92.17%


In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, questions, answers in val_loader:

        images = images.to(device)
        questions = questions.to(device)
        answers = answers.to(device)

        logits = model(
            images,
            questions
        )

        predictions = torch.argmax(
            logits,
            dim=1
        )

        correct += (
            predictions == answers
        ).sum().item()

        total += answers.size(0)

val_accuracy = 100 * correct / total

print(
    f"Validation Accuracy: {val_accuracy:.2f}%"
)

Validation Accuracy: 88.94%
